# Tarea B — Temperatura, top-p y entropía de la distribución de salida

## Parte 1 — Softmax con temperatura

In [1]:
# Subtarea T1: Definir el vector de logits de los seis tokens t0..t5 (2.0, 1.0, 0.5, 0.2, -1.0, -3.0) e implementar en NumPy la softmax con temperatura de forma numé
# -*- coding: utf-8 -*-
"""
SUBTAREA T1 — Softmax con temperatura y entropía en bits
Tarea B — Temperatura, top-p y entropía de la distribución de salida (Parte 1)

Define el vector de logits de los seis tokens t0..t5, implementa la softmax
con temperatura de forma numéricamente estable (restando el máximo), calcula
la distribución para T = 0.5, 1 y 2, y la entropía en bits de cada una.
Presenta una tabla con cuatro decimales y guarda todas las cifras en
"resultados.json". No se requiere figura.
"""

import json
import numpy as np

# ----------------------------------------------------------------------
# 1. Datos del enunciado
# ----------------------------------------------------------------------
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)

# ----------------------------------------------------------------------
# 2. Softmax con temperatura, numéricamente estable
#    p_i = exp(z_i / T) / sum_j exp(z_j / T)
#    Estabilidad: se resta el máximo de los logits escalados antes de exp,
#    lo que evita desbordamientos sin cambiar el resultado (factor común).
# ----------------------------------------------------------------------
def softmax_con_temperatura(z, T):
    z = np.asarray(z, dtype=float)
    if T <= 0:
        raise ValueError("La temperatura T debe ser positiva.")
    escalados = z / T
    escalados = escalados - np.max(escalados)  # resta del máximo (estabilidad)
    exps = np.exp(escalados)
    return exps / np.sum(exps)

# ----------------------------------------------------------------------
# 3. Entropía en bits: H = -sum_i p_i * log2(p_i)
# ----------------------------------------------------------------------
def entropia_bits(p):
    p = np.asarray(p, dtype=float)
    mascara = p > 0  # los términos con p_i = 0 aportan 0 a la entropía
    return float(-np.sum(p[mascara] * np.log2(p[mascara])))

# ----------------------------------------------------------------------
# 4. Cálculo para T = 0.5, 1 y 2
# ----------------------------------------------------------------------
temperaturas = [0.5, 1.0, 2.0]
distribuciones = {}
entropias = {}
sumas = {}

for T in temperaturas:
    p = softmax_con_temperatura(logits, T)
    distribuciones[T] = p
    entropias[T] = entropia_bits(p)
    sumas[T] = float(np.sum(p))  # verificación: debe ser 1.0

# ----------------------------------------------------------------------
# 5. Tabla con cuatro decimales (distribuciones y entropías)
# ----------------------------------------------------------------------
print("Tabla: distribuciones softmax con temperatura y entropia en bits")
print("-" * 70)
encabezado = f"{'T':<6}" + "".join(f"{tok:>9}" for tok in tokens) + f"{'H (bits)':>12}"
print(encabezado)
print("-" * 70)
for T in temperaturas:
    fila = f"{T:<6}" + "".join(f"{p:>9.4f}" for p in distribuciones[T])
    fila += f"{entropias[T]:>12.4f}"
    print(fila)
print("-" * 70)

# ----------------------------------------------------------------------
# 6. Guardar TODAS las cifras calculadas en resultados.json
# ----------------------------------------------------------------------
resultados = {
    "tokens": tokens,
    "logits": [float(z) for z in logits],
    "temperaturas": [float(T) for T in temperaturas],
    "distribuciones_softmax": {
        f"T={T}": {tok: float(p) for tok, p in zip(tokens, distribuciones[T])}
        for T in temperaturas
    },
    "suma_distribuciones": {f"T={T}": float(sumas[T]) for T in temperaturas},
    "entropia_bits": {f"T={T}": float(entropias[T]) for T in temperaturas},
}

with open("resultados.json", "w") as f:
    json.dump(resultados, f, indent=2)

# ----------------------------------------------------------------------
# 7. Cifras principales con 4 decimales
# ----------------------------------------------------------------------
print("\nCifras principales:")
for T in temperaturas:
    probs_str = ", ".join(f"{tok}={p:.4f}" for tok, p in zip(tokens, distribuciones[T]))
    print(f"T={T}: p = [{probs_str}]  |  H = {entropias[T]:.4f} bits  |  suma = {sumas[T]:.4f}")

print("\nResultados guardados en 'resultados.json'.")


Tabla: distribuciones softmax con temperatura y entropia en bits
----------------------------------------------------------------------
T            t0       t1       t2       t3       t4       t5    H (bits)
----------------------------------------------------------------------
0.5      0.8231   0.1114   0.0410   0.0225   0.0020   0.0000      0.9147
1.0      0.5516   0.2029   0.1231   0.0912   0.0275   0.0037      1.7998
2.0      0.3583   0.2173   0.1693   0.1457   0.0800   0.0294      2.2888
----------------------------------------------------------------------

Cifras principales:
T=0.5: p = [t0=0.8231, t1=0.1114, t2=0.0410, t3=0.0225, t4=0.0020, t5=0.0000]  |  H = 0.9147 bits  |  suma = 1.0000
T=1.0: p = [t0=0.5516, t1=0.2029, t2=0.1231, t3=0.0912, t4=0.0275, t5=0.0037]  |  H = 1.7998 bits  |  suma = 1.0000
T=2.0: p = [t0=0.3583, t1=0.2173, t2=0.1693, t3=0.1457, t4=0.0800, t5=0.0294]  |  H = 2.2888 bits  |  suma = 1.0000

Resultados guardados en 'resultados.json'.


Implementé la softmax con temperatura de forma numéricamente estable: antes de exponenciar resto el logit máximo, es decir, p_i = exp((z_i − z_max)/T) / Σ_j exp((z_j − z_max)/T). Restar el máximo no cambia el resultado (el factor común se cancela al normalizar), pero evita que las exponenciales se desborden. Con los seis logits calculé la distribución para T = 0.5, T = 1 y T = 2, y la entropía en bits de cada una; la tabla con cuatro decimales la imprime la celda de código.

Lo que se observa: al subir la temperatura la masa se reparte desde el token líder hacia el resto. t0 pasa de 0.8231 con T = 0.5 a 0.5516 con T = 1 y a 0.3583 con T = 2, mientras que t5 sube de 0.0000 a 0.0037 y luego a 0.0294. En consecuencia la entropía crece: 0.9147 bits con T = 0.5, 1.7998 bits con T = 1 y 2.2888 bits con T = 2. Las tres distribuciones suman exactamente 1.

## Parte 2 — Muestreo de núcleo (top-p)

In [2]:
# Subtarea T2: Sobre la distribución de T=1 de la Parte 1, aplicar muestreo de núcleo top-p con p=0.9: ordenar los tokens por probabilidad descendente, conservar el 
# -*- coding: utf-8 -*-
"""
SUBTAREA T2 — Muestreo de núcleo (top-p) con p = 0.9 sobre la distribución de T = 1.

Pasos:
1) Recalcula de forma independiente la distribución softmax con temperatura T = 1
   (Parte 1) a partir de los logits del enunciado.
2) Ordena los tokens por probabilidad descendente.
3) Conserva el conjunto mínimo de tokens cuya probabilidad acumulada alcanza p = 0.9
   (el token que hace cruzar el umbral se incluye).
4) Renormaliza las probabilidades de los tokens supervivientes.

Guarda todas las cifras en 'resultados.json' e imprime las principales con 4 decimales.
No se requiere figura.
"""

import json
import numpy as np

# ---------------------------------------------------------------------
# Datos del enunciado (Parte 1)
# ---------------------------------------------------------------------
TOKENS = ["t0", "t1", "t2", "t3", "t4", "t5"]
LOGITS = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
T = 1.0   # temperatura de la distribución de la Parte 1 sobre la que se aplica top-p
P = 0.9   # umbral del núcleo (nucleus sampling)

# ---------------------------------------------------------------------
# Softmax con temperatura, numéricamente estable (recálculo de la Parte 1)
# ---------------------------------------------------------------------
def softmax_con_temperatura(logits, temperatura):
    z = np.asarray(logits, dtype=float) / float(temperatura)
    z = z - np.max(z)                     # estabilidad numérica: restar el máximo
    e = np.exp(z)
    return e / np.sum(e)

p_T1 = softmax_con_temperatura(LOGITS, T)

# ---------------------------------------------------------------------
# Top-p (nucleus sampling) con p = 0.9
# ---------------------------------------------------------------------
# 1) Ordenar los tokens por probabilidad descendente
orden = np.argsort(-p_T1, kind="stable")
probs_ord = p_T1[orden]
tokens_ord = [TOKENS[i] for i in orden]

# 2) Probabilidad acumulada en ese orden
acum = np.cumsum(probs_ord)

# 3) Conjunto mínimo cuya probabilidad acumulada alcanza p
#    (se incluye el token con el que la acumulada cruza el umbral)
k = 0
acc = 0.0
for prob in probs_ord:
    k += 1
    acc += float(prob)
    if acc >= P:
        break

idx_nucleo = orden[:k]
tokens_nucleo = [TOKENS[i] for i in idx_nucleo]
tokens_fuera = [TOKENS[i] for i in orden[k:]]
suma_nucleo = acc  # probabilidad acumulada del núcleo antes de renormalizar

# 4) Renormalizar
p_renorm_nucleo = p_T1[idx_nucleo] / suma_nucleo

# Distribución renormalizada en el orden original t0..t5 (0.0 para descartados)
dist_renorm = {tok: 0.0 for tok in TOKENS}
for i, tok in zip(idx_nucleo, tokens_nucleo):
    dist_renorm[tok] = float(p_T1[i] / suma_nucleo)

# Cifras auxiliares
probs_T1 = {tok: float(v) for tok, v in zip(TOKENS, p_T1)}
acum_dict = {tok: float(a) for tok, a in zip(tokens_ord, acum)}
probs_ord_list = [float(v) for v in probs_ord]

def entropia_bits(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return float(-np.sum(p * np.log2(p)))

H_T1 = entropia_bits(p_T1)
H_renorm = entropia_bits(p_renorm_nucleo)
suma_renorm = float(sum(dist_renorm.values()))

# ---------------------------------------------------------------------
# Guardar TODAS las cifras en resultados.json (fusionando con lo previo)
# ---------------------------------------------------------------------
resultados_T2 = {
    "subtarea": "T2: top-p (nucleus) con p=0.9 sobre la distribucion de T=1",
    "tokens": list(TOKENS),
    "logits": [float(z) for z in LOGITS],
    "temperatura": float(T),
    "p": float(P),
    "distribucion_T1": probs_T1,
    "tokens_ordenados_desc": tokens_ord,
    "probs_ordenadas_desc": probs_ord_list,
    "prob_acumulada_orden_desc": acum_dict,
    "num_tokens_nucleo": int(k),
    "tokens_sobrevivientes": tokens_nucleo,
    "tokens_descartados": tokens_fuera,
    "suma_prob_nucleo_antes_renorm": float(suma_nucleo),
    "distribucion_renormalizada": dist_renorm,
    "suma_distribucion_renormalizada": suma_renorm,
    "entropia_bits_T1_original": H_T1,
    "entropia_bits_tras_top_p": H_renorm,
}

try:
    with open("resultados.json", "r") as f:
        previo = json.load(f)
    if not isinstance(previo, dict):
        previo = {}
except (FileNotFoundError, json.JSONDecodeError):
    previo = {}

previo["T2_top_p"] = resultados_T2

with open("resultados.json", "w") as f:
    json.dump(previo, f, indent=2, ensure_ascii=False)

# ---------------------------------------------------------------------
# Reporte
# ---------------------------------------------------------------------
print("Top-p (nucleus sampling) con p = 0.9 sobre la distribucion de T = 1")
print("------------------------------------------------------------------")
print(f"{'token':>6} {'p(T=1)':>10} {'acumulada':>10}   nucleo")
print("------------------------------------------------------------------")
en_nucleo = set(tokens_nucleo)
for tok in tokens_ord:
    marca = "si" if tok in en_nucleo else "no"
    print(f"{tok:>6} {probs_T1[tok]:>10.4f} {acum_dict[tok]:>10.4f}   {marca}")
print("------------------------------------------------------------------")
print(f"Tokens sobrevivientes (conjunto minimo con acumulada >= {P:.1f}): {tokens_nucleo}")
print(f"Probabilidad acumulada del nucleo antes de renormalizar: {suma_nucleo:.4f}")
print(f"Tokens descartados: {tokens_fuera}")
print()
print("Distribucion renormalizada:")
for tok in TOKENS:
    print(f"  {tok}: {dist_renorm[tok]:.4f}")
print(f"Suma: {suma_renorm:.4f}")
print()
print("Cifras principales:")
print(f"  Nucleo (p={P:.1f}): {tokens_nucleo} "
      f"(k={k} tokens, acumulada original = {suma_nucleo:.4f})")
print("  Distribucion renormalizada: "
      + ", ".join(f"{tok}={dist_renorm[tok]:.4f}" for tok in tokens_nucleo))
print(f"  Entropia T=1 original: {H_T1:.4f} bits | Entropia tras top-p: {H_renorm:.4f} bits")
print()
print("Resultados guardados en 'resultados.json'.")


Top-p (nucleus sampling) con p = 0.9 sobre la distribucion de T = 1
------------------------------------------------------------------
 token     p(T=1)  acumulada   nucleo
------------------------------------------------------------------
    t0     0.5516     0.5516   si
    t1     0.2029     0.7546   si
    t2     0.1231     0.8776   si
    t3     0.0912     0.9688   si
    t4     0.0275     0.9963   no
    t5     0.0037     1.0000   no
------------------------------------------------------------------
Tokens sobrevivientes (conjunto minimo con acumulada >= 0.9): ['t0', 't1', 't2', 't3']
Probabilidad acumulada del nucleo antes de renormalizar: 0.9688
Tokens descartados: ['t4', 't5']

Distribucion renormalizada:
  t0: 0.5694
  t1: 0.2095
  t2: 0.1270
  t3: 0.0941
  t4: 0.0000
  t5: 0.0000
Suma: 1.0000

Cifras principales:
  Nucleo (p=0.9): ['t0', 't1', 't2', 't3'] (k=4 tokens, acumulada original = 0.9688)
  Distribucion renormalizada: t0=0.5694, t1=0.2095, t2=0.1270, t3=0.0941
  Entr

Ordené los tokens de mayor a menor probabilidad usando la distribución de T = 1 y fui acumulando: t0 acumula 0.5516, con t1 llega a 0.7546, con t2 a 0.8776 y con t3 a 0.9688. Como después de t2 la acumulada todavía no alcanza 0.9, el conjunto mínimo que la alcanza es {t0, t1, t2, t3}; t4 y t5 quedan fuera. El núcleo se pasa del umbral (suma 0.9688 en lugar de 0.9) porque los tokens se conservan enteros: no se puede "cortar" a t3 a la mitad.

La distribución renormalizada (cada probabilidad del núcleo dividida entre 0.9688) queda t0 = 0.5694, t1 = 0.2095, t2 = 0.1270 y t3 = 0.0941, con t4 = t5 = 0. El efecto del recorte se ve en la entropía: baja de 1.7998 a 1.6341 bits, porque se eliminó la cola de baja probabilidad y la masa restante quedó un poco más concentrada.

## Parte 3 — Comprobación empírica

In [3]:
# Subtarea T3: Tomar 10000 muestras de la distribución renormalizada de la Parte 2 con numpy.random.default_rng(0). Comparar frecuencias observadas vs probabilidades
# -*- coding: utf-8 -*-
"""
SUBTAREA T3 — Comprobación empírica del muestreo top-p (Tarea B, Parte 3)

Pasos:
  1) Recalcula la distribución softmax con T = 1 (Parte 1) a partir de los logits.
  2) Recalcula el muestreo de núcleo top-p con p = 0.9 y la renormalización (Parte 2).
  3) Toma 10000 muestras de la distribución renormalizada con numpy.random.default_rng(0).
  4) Compara frecuencias observadas vs probabilidades teóricas en un gráfico de barras (PNG).
  5) Calcula la divergencia KL (en bits) de la distribución empírica respecto de la teórica.
  6) Guarda todas las cifras en 'resultados.json' e imprime las principales con 4 decimales.
"""

import json
import math

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# 1. Datos del enunciado
# ----------------------------------------------------------------------
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)

# ----------------------------------------------------------------------
# 2. Parte 1 (recalculada): softmax con temperatura, numéricamente estable
# ----------------------------------------------------------------------
def softmax_con_temperatura(z, T):
    z = np.asarray(z, dtype=float)
    escalados = z / float(T)
    escalados = escalados - np.max(escalados)  # truco de estabilidad numérica
    expz = np.exp(escalados)
    return expz / np.sum(expz)

T = 1.0
p_T1 = softmax_con_temperatura(logits, T)

# ----------------------------------------------------------------------
# 3. Parte 2 (recalculada): top-p (nucleus) con p = 0.9 y renormalización
# ----------------------------------------------------------------------
p_objetivo = 0.9
orden_desc = np.argsort(-p_T1, kind="stable")  # índices por probabilidad descendente

acum = 0.0
indices_nucleo = []
for i in orden_desc:
    indices_nucleo.append(int(i))
    acum += float(p_T1[i])
    if acum >= p_objetivo:  # conjunto mínimo cuya acumulada alcanza 0.9
        break

suma_nucleo = float(np.sum(p_T1[indices_nucleo]))

q_teorica = np.zeros(len(tokens), dtype=float)
q_teorica[indices_nucleo] = p_T1[indices_nucleo] / suma_nucleo  # renormalización

# ----------------------------------------------------------------------
# 4. Parte 3: 10000 muestras de la distribución renormalizada con default_rng(0)
# ----------------------------------------------------------------------
N = 10000
semilla = 0
rng = np.random.default_rng(semilla)

q_muestreo = q_teorica / np.sum(q_teorica)  # garantiza suma exactamente 1 para rng.choice
muestras = rng.choice(len(tokens), size=N, p=q_muestreo)

conteos = np.bincount(muestras, minlength=len(tokens)).astype(int)
frecuencias_obs = conteos / float(N)

# ----------------------------------------------------------------------
# 5. Divergencia KL (en bits) de la empírica respecto de la teórica
#    KL(q || p) = sum_i q_i * log2(q_i / p_i),  con la convención 0*log(0/.) = 0
# ----------------------------------------------------------------------
def kl_bits(q, p):
    kl = 0.0
    for qi, pi in zip(q, p):
        if qi > 0.0:
            if pi <= 0.0:
                return float("inf")
            kl += qi * math.log2(qi / pi)
    return float(kl)

def entropia_bits(p):
    h = 0.0
    for pi in p:
        if pi > 0.0:
            h -= pi * math.log2(pi)
    return float(h)

kl_emp_teo = kl_bits(frecuencias_obs, q_teorica)
H_teorica = entropia_bits(q_teorica)
H_empirica = entropia_bits(frecuencias_obs)

diferencias = frecuencias_obs - q_teorica
error_abs_max = float(np.max(np.abs(diferencias)))
tokens_sin_muestras = [tokens[i] for i in range(len(tokens)) if conteos[i] == 0]

# ----------------------------------------------------------------------
# 6. Figura: barras de frecuencias observadas vs probabilidades teóricas
# ----------------------------------------------------------------------
nombre_figura = "t3_frecuencias_vs_teoricas.png"
x = np.arange(len(tokens))
ancho = 0.38

fig, ax = plt.subplots(figsize=(8.5, 5.2))
b_obs = ax.bar(x - ancho / 2.0, frecuencias_obs, ancho,
               label="Frecuencia observada (10000 muestras)",
               color="#4C72B0", edgecolor="black", linewidth=0.6)
b_teo = ax.bar(x + ancho / 2.0, q_teorica, ancho,
               label="Probabilidad teórica (top-p p=0.9 renormalizada)",
               color="#DD8452", edgecolor="black", linewidth=0.6)

for barras in (b_obs, b_teo):
    for b in barras:
        h = b.get_height()
        ax.annotate(f"{h:.4f}",
                    xy=(b.get_x() + b.get_width() / 2.0, h),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_xlabel("Token")
ax.set_ylabel("Probabilidad / frecuencia")
ax.set_ylim(0, max(np.max(frecuencias_obs), np.max(q_teorica)) * 1.18)
ax.set_title("Parte 3 — Frecuencias observadas vs probabilidades teóricas\n"
             f"top-p (p=0.9) sobre T=1, N={N}, semilla={semilla} · "
             f"$D_{{KL}}$(empírica||teórica) = {kl_emp_teo:.4f} bits")
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.3, linestyle="--")

fig.tight_layout()
fig.savefig(nombre_figura, dpi=150)
plt.close(fig)

# ----------------------------------------------------------------------
# 7. Guardar TODAS las cifras en resultados.json (objeto, floats sin redondear)
# ----------------------------------------------------------------------
cifras_t3 = {
    "subtarea": ("T3: comprobacion empirica — 10000 muestras de la distribucion renormalizada "
                 "top-p (p=0.9 sobre T=1) con numpy.random.default_rng(0); comparacion de "
                 "frecuencias observadas vs probabilidades teoricas y divergencia KL en bits"),
    "tokens": tokens,
    "logits": [float(z) for z in logits],
    "temperatura": float(T),
    "p_top_p": float(p_objetivo),
    "distribucion_T1_original": {tokens[i]: float(p_T1[i]) for i in range(len(tokens))},
    "tokens_nucleo": [tokens[i] for i in indices_nucleo],
    "num_tokens_nucleo": int(len(indices_nucleo)),
    "suma_prob_nucleo_antes_renorm": float(suma_nucleo),
    "distribucion_renormalizada_teorica": {tokens[i]: float(q_teorica[i]) for i in range(len(tokens))},
    "n_muestras": int(N),
    "semilla": int(semilla),
    "generador": "numpy.random.default_rng(0)",
    "conteos_observados": {tokens[i]: int(conteos[i]) for i in range(len(tokens))},
    "frecuencias_observadas": {tokens[i]: float(frecuencias_obs[i]) for i in range(len(tokens))},
    "diferencia_obs_menos_teorica": {tokens[i]: float(diferencias[i]) for i in range(len(tokens))},
    "error_absoluto_maximo": float(error_abs_max),
    "suma_frecuencias_observadas": float(np.sum(frecuencias_obs)),
    "tokens_sin_muestras": tokens_sin_muestras,
    "kl_empirica_respecto_teorica_bits": float(kl_emp_teo),
    "entropia_bits_distribucion_teorica": float(H_teorica),
    "entropia_bits_distribucion_empirica": float(H_empirica),
    "figura": nombre_figura,
}

try:
    with open("resultados.json", "r", encoding="utf-8") as f:
        previo = json.load(f)
    if not isinstance(previo, dict):
        previo = {}
except (FileNotFoundError, OSError, ValueError):
    previo = {}

previo["T3_comprobacion_empirica"] = cifras_t3

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(previo, f, indent=2, ensure_ascii=False)

# ----------------------------------------------------------------------
# 8. Impresión de las cifras principales (4 decimales)
# ----------------------------------------------------------------------
print("Parte 3 — Comprobacion empirica del muestreo top-p")
print("--------------------------------------------------")
print(f"Nucleo top-p (p=0.9): {[tokens[i] for i in indices_nucleo]} "
      f"(suma original = {suma_nucleo:.4f})")
print(f"Muestreo: N = {N} muestras, generador numpy.random.default_rng({semilla})")
print("--------------------------------------------------")
print("token   conteo   frec_obs   p_teorica    diff")
for i in range(len(tokens)):
    print(f"  {tokens[i]}  {conteos[i]:8d}   {frecuencias_obs[i]:.4f}     "
          f"{q_teorica[i]:.4f}   {diferencias[i]:+.4f}")
print("--------------------------------------------------")
print(f"D_KL(empirica || teorica) = {kl_emp_teo:.4f} bits")
print(f"Entropia teorica = {H_teorica:.4f} bits | Entropia empirica = {H_empirica:.4f} bits")
print(f"Error absoluto maximo |frec_obs - p_teorica| = {error_abs_max:.4f}")
print(f"Tokens sin muestras: {tokens_sin_muestras if tokens_sin_muestras else 'ninguno'}")
print(f"Figura guardada en: {nombre_figura}")
print("Resultados guardados en 'resultados.json'.")


Parte 3 — Comprobacion empirica del muestreo top-p
--------------------------------------------------
Nucleo top-p (p=0.9): ['t0', 't1', 't2', 't3'] (suma original = 0.9688)
Muestreo: N = 10000 muestras, generador numpy.random.default_rng(0)
--------------------------------------------------
token   conteo   frec_obs   p_teorica    diff
  t0      5682   0.5682     0.5694   -0.0012
  t1      2093   0.2093     0.2095   -0.0002
  t2      1296   0.1296     0.1270   +0.0026
  t3       929   0.0929     0.0941   -0.0012
  t4         0   0.0000     0.0000   +0.0000
  t5         0   0.0000     0.0000   +0.0000
--------------------------------------------------
D_KL(empirica || teorica) = 0.0001 bits
Entropia teorica = 1.6341 bits | Entropia empirica = 1.6362 bits
Error absoluto maximo |frec_obs - p_teorica| = 0.0026
Tokens sin muestras: ['t4', 't5']
Figura guardada en: t3_frecuencias_vs_teoricas.png
Resultados guardados en 'resultados.json'.


Sorteé N = 10 000 muestras de la distribución renormalizada con numpy.random.default_rng(0) y comparé las frecuencias observadas con las probabilidades teóricas; la celda de código imprime la tabla y genera el gráfico. Los conteos fueron t0 = 5682, t1 = 2093, t2 = 1296 y t3 = 929, es decir, frecuencias de 0.5682, 0.2093, 0.1296 y 0.0929. La mayor desviación absoluta frente a la teórica es 0.0026 y ocurre en t2. t4 y t5 no reciben ninguna muestra, como debe ser: tras el top-p su probabilidad es cero.

La divergencia KL de la distribución empírica respecto de la teórica da 0.0001 bits, prácticamente cero. En la misma línea, la entropía empírica (1.6362 bits) queda muy cerca de la teórica (1.6341 bits). Con 10 000 muestras el error de muestreo esperado es pequeño y el resultado lo confirma: el muestreador reproduce la distribución del núcleo.

![Frecuencias observadas frente a probabilidades teóricas del núcleo top-p (10 000 muestras, semilla 0)](T3_t3_frecuencias_vs_teoricas.png)

## Parte 4 — Pregunta conceptual

**T → 0: la distribución se concentra en el argmax.** La temperatura divide los logits antes de exponenciar, así que su efecto real es sobre las diferencias entre logits: p_i ∝ exp((z_i − z_max)/T). Cuando T → 0⁺, esa diferencia se va a −∞ para todo token distinto del de mayor logit, su exponencial se apaga y toda la masa queda en el argmax (aquí t0). En el límite, la softmax es un one-hot: probabilidad 1 para t0 y 0 para el resto. La Parte 1 muestra ese camino: al bajar T de 1 a 0.5, t0 sube de 0.5516 a 0.8231 y la entropía cae de 1.7998 a 0.9147 bits. Como la entropía de una distribución puntual es 0 bits y en los datos H decrece al bajar T (2.2888 → 1.7998 → 0.9147), el límite T → 0 implica H → 0: cero incertidumbre.

**T → ∞: la distribución tiende a la uniforme.** En el otro extremo, z_i/T → 0 para todos los tokens, todas las exponenciales valen lo mismo y la distribución converge a la uniforme sobre el vocabulario: 1/6 para cada token. La entropía tiende entonces al máximo posible para seis opciones, que es log₂(6) bits. Los datos lo insinúan: con T = 2 la distribución se aplana (t0 baja a 0.3583 y t5 sube a 0.0294, frente a 0.5516 y 0.0037 con T = 1) y H sube a 2.2888 bits, todavía debajo del máximo pero claramente acercándose. La temperatura funciona, pues, como un dial entre dos extremos: elección determinista (T → 0) y elección totalmente azarosa (T → ∞).

**Por qué T → 0 equivale a greedy.** La decodificación voraz elige en cada paso el token de mayor logit, argmax_i z_i, sin sortear nada. Si en lugar de argmax se muestrea de la softmax con T → 0, el único token con probabilidad distinta de cero es precisamente ese argmax, de modo que la muestra sale argmax con probabilidad 1: muestrear de un one-hot y tomar el argmax son la misma operación. La evidencia son las entropías de la Parte 1: greedy corresponde a una distribución de entropía 0 bits, y las entropías calculadas decrecen hacia ese valor conforme baja la temperatura (1.7998 bits con T = 1, 0.9147 bits con T = 0.5). En el límite no queda incertidumbre que el azar resuelva: el "muestreo" degenera en elegir siempre el token más probable, que es exactamente lo que hace greedy.